# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

|   |   |
|---|---|
| **Title** | Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells |
| **Identifier** | 10.71728/senscience.vq4a-28xa |
| **License** | [Open Data Commons BY 1.0](https://opendatacommons.org/licenses/by/1-0/) |


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print basic dataset metadata
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Identifier: {dataset.metadata.identifier}\n")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s as defined in the schema. 
All entities are referenced explicitly by their `@id` for robust programmatic access.


In [ ]:
# List available record sets and their fields using @id referencing
print("Available Record Sets (by @id):")
rs_ids = [rset['@id'] for rset in dataset.metadata.record_sets]
for rset in dataset.metadata.record_sets:
    print(f"  - @id: {rset['@id']}, name: {rset.get('name', '')}")

# Show the fields/columns for each record set by @id
for rset in dataset.metadata.record_sets:
    print(f"\nRecord Set: {rset['@id']}")
    fields = [f['@id'] for f in rset['fields']]
    print("  Field @ids:")
    for f in rset['fields']:
        print(f"    - {f['@id']}: {f.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, all entities (record sets and fields) are referenced by their `@id`, and the extraction is performed into Pandas DataFrames so you can interact and analyze the data efficiently.

In [ ]:
# Define which record sets to load (by @id)
record_set_ids = [rset['@id'] for rset in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Record set '{rs_id}' - loaded {len(df)} records. Columns: {list(df.columns)}\n")
# For demonstration: show the available columns for the first record set
if record_set_ids:
    print(f"First record set columns: {dataframes[record_set_ids[0]].columns.tolist()}")
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on a numeric criterion, normalizing a numeric field, and grouping by another field. 

💡 All fields referenced by their `@id`. Please check the available numeric and categorical fields above (step 2/3) and adjust the field IDs below as appropriate.


In [ ]:
# Example: Select a numeric field and group field for the first record set
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Identify sample numeric and grouping field (change accordingly to your data)
print("Sample columns (by @id):", df.columns.tolist())

# Example placeholder: replace with your chosen fields by @id
numeric_field_id = df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns) > 0 else None
group_field_id = df.columns[1] if len(df.columns) > 1 else None

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (by @id):")
    display(filtered_df.head())

    filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized field '{numeric_field_id}' (mean=0, std=1) for the filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a field if categorical/grouping field exists
    if group_field_id is not None and not pd.api.types.is_numeric_dtype(df[group_field_id]):
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by '{group_field_id}' (by @id):")
        display(grouped_df.head())
else:
    print("No numeric field found to analyze.")

## 5. Visualization
Visualize data distributions or relationships using matplotlib or seaborn. Adjust the fields referenced below according to your dataset's field `@id`s.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field (if available)
if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Optionally, visualize relations (scatter) if another numeric field exists
    other_numeric_fields = [col for col in df.select_dtypes('number').columns if col != numeric_field_id]
    if other_numeric_fields:
        other_field = other_numeric_fields[0]
        plt.figure(figsize=(7,5))
        sns.scatterplot(x=df[numeric_field_id], y=df[other_field])
        plt.title(f"'{numeric_field_id}' vs '{other_field}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel(other_field)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion

- This notebook demonstrated loading and processing the FAIR² dataset using `mlcroissant`.
- All schema elements (record sets and fields) were accessed using their `@id`, ensuring robust reference throughout data extraction and analysis.
- EDA and simple visualizations were performed to gain insights into the content and structure of the loaded record sets.

➡️ You can now use this notebook as a template to perform targeted analysis, training, or further integration for your own ML workflows.